In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS football.gold

In [0]:
%sql
CREATE OR REPLACE VIEW football.gold.team_match_stats
COMMENT 'One row per team per match: shots, goals, xG, passes, pressures'
AS
SELECT
  match_id,
  team,
  COUNT_IF(event_type = 'Shot')                            AS shots,
  COUNT_IF(event_type = 'Shot' AND shot_outcome = 'Goal')  AS goals,
  ROUND(SUM(COALESCE(xg, 0)), 2)                           AS xg,
  COUNT_IF(event_type = 'Pass')                            AS passes,
  COUNT_IF(event_type = 'Pressure')                        AS pressures,
  MAX(minute)                                              AS last_minute
FROM football.silver.events
WHERE team IS NOT NULL
GROUP BY match_id, team

In [0]:
%sql
CREATE OR REPLACE VIEW football.gold.xg_timeline
COMMENT 'Cumulative expected goals by team and match minute'
AS
SELECT
  match_id,
  team,
  minute,
  ROUND(SUM(SUM(COALESCE(xg, 0))) OVER (
        PARTITION BY match_id, team ORDER BY minute), 2) AS cum_xg
FROM football.silver.events
WHERE event_type = 'Shot'
GROUP BY match_id, team, minute

In [0]:
%sql
CREATE OR REPLACE VIEW football.gold.shot_map
COMMENT 'Every shot with pitch location (x 0-120, y 0-80), xG and outcome'
AS
SELECT
  match_id, team, player, minute, x, y,
  ROUND(xg, 3) AS xg,
  shot_outcome
FROM football.silver.events
WHERE event_type = 'Shot' AND x IS NOT NULL

In [0]:
%sql
CREATE OR REPLACE VIEW football.gold.player_form
COMMENT 'Shots, goals and xG per player'
AS
SELECT
  player, team,
  COUNT(DISTINCT match_id)                                 AS matches,
  COUNT_IF(event_type = 'Shot')                            AS shots,
  COUNT_IF(event_type = 'Shot' AND shot_outcome = 'Goal')  AS goals,
  ROUND(SUM(COALESCE(xg, 0)), 2)                           AS xg
FROM football.silver.events
WHERE player IS NOT NULL
GROUP BY player, team

In [0]:
%sql
SELECT * FROM football.gold.team_match_stats

In [0]:
%sql
SELECT * FROM football.gold.xg_timeline ORDER BY team, minute

In [0]:
%sql
SELECT player, team, shots, goals, xg
FROM football.gold.player_form
WHERE shots > 0
ORDER BY xg DESC
LIMIT 10